In [121]:
import psycopg2
import pandas as pd
import re
import os
import warnings
import zipfile

# Suppress unnecessary warnings
warnings.filterwarnings('ignore', category=UserWarning)

warnings.filterwarnings('ignore', category=SyntaxWarning)

import sys
UTILS_FOLDER = r"D:\github\My_function"
sys.path.append(UTILS_FOLDER)

# Now import your functions
from ret_finding import eric_air, lte_cell_normalized, get_site_name, hwret, eric_non_air


In [122]:
cluster_name = 'BMA0319'
week_name = 'wk2521'
folder_name = cluster_name.split('_')[0]

INPUT_FILE_PATH = f'D:/D&T Project/CR Preparing/{folder_name}/Tuning_cell_list_{cluster_name}.csv'
OUTPUT_BASE_DIR = f'D:/D&T Project/CR Preparing/'


HOST = "localhost"
PORT = "5432"
DATABASE = "postgres"
USER = "postgres"
PASSWORD = "aaa"

sql_lte = f'lte_{week_name}'
sql_nr = f'nr_{week_name}'



### ===== Database Connection Setup =====

In [123]:

def connect_postgres(host, port, database, user, password):
    try:
        conn = psycopg2.connect(
            host=host,
            port=port,
            database=database,
            user=user,
            password=password
        )
        print("Connected to PostgreSQL database!")
        return conn
    except Exception as e:
        print(f"Error connecting to PostgreSQL: {e}")
        return None

### ===== Data Extraction Function =====

In [124]:
# ===== Site Name Extraction =====
def extract_site_name(cell_name):
    match = re.search(r'[A-Z]{3,4}\d{3,4}', cell_name)
    return match.group(0) if match else "No Site Name"

# ===== Load Input Cell List =====
def load_cell_list(input_file_path):
    df = pd.read_csv(input_file_path)
    df.columns = [col.lower() for col in df.columns]
    df['cell name'] = df['cell name'].str.strip()
    df['site_name_1'] = df['cell name'].apply(extract_site_name)
    return df

In [125]:
# ===== Generate WHERE Clause =====
def generate_where_clause(site_ids):
    site_ids_str = "', '".join(site_ids)
    where_clause = f"site IN ('{site_ids_str}')"
    where_clause_1 = f"left(nodeid,7) IN ('{site_ids_str}')"
    where_clause_2 = f"site_name IN ('{site_ids_str}')"
    
    return where_clause, where_clause_1, where_clause_2

# ===== Fetch Data from PostgreSQL =====
def fetch_data(query, conn):
    return pd.read_sql_query(query, conn)

def tuning_band_logic(system):
    if system in ['L1800', 'L2100']:
        return 'MB'
    elif system in ['L700', 'L900','NR700']:
        return 'LB'
    elif system in ['L2600', 'NR2600']:
        return '2600'
    elif system in ['L2300']:
        return system
    else:
        return 'Unknown'  



In [126]:
df_cell = load_cell_list(INPUT_FILE_PATH)
site_ids = df_cell['site_name_1'].unique()
where_clause, where_clause_1, where_clause_2 = generate_where_clause(site_ids)


In [127]:
where_clause_2

"site_name IN ('PTT7028', 'PTT7102', 'PTTC091', 'PTT8518', 'PTT0243', 'PTT2018', 'PTT7982', 'PTTA093', 'PTT7140', 'PTT1020', 'PTT3935', 'PTT7981', 'PTTA078', 'PTTC057', 'PTT2883', 'PTT8610', 'PTT6320', 'PTT3236', 'BKK6661', 'PTTC058', 'PTT0777', 'PTT7983', 'PTTC152', 'PTTA088', 'PTTC042', 'PTTC054', 'PTT7183', 'PTT3868', 'PTT1185', 'PTT1222', 'PTT2918', 'PTT7223', 'PTTC145', 'PTTC148', 'PTT1146', 'PTT8620', 'PTT1175', 'PTT7300', 'PTT6700', 'PTTC047', 'PTT2203', 'PTT8221')"

In [128]:
df_cell

,cell name,cluster name,site_name_1
0,L23-PTT7028-1D,BMA0319,PTT7028
1,L23-PTT7028-3A,BMA0319,PTT7028
2,L23-PTT7102-1D,BMA0319,PTT7102
3,L23-PTTC091-1C,BMA0319,PTTC091
4,L23-PTT8518-1C,BMA0319,PTT8518
...,...,...,...
215,PTT7140T_2NB01_S52,BMA0319,PTT7140
216,PTTA093E_2NB01_S52,BMA0319,PTTA093
217,PTTC145T_2NB01_S03,BMA0319,PTTC145
218,PTT8221R_1NB02_R01,BMA0319,PTT8221


### ====== COMMON SQL =====

In [129]:
query_lte = f"""
SELECT site, site_id, cell_name, system, sector_name, antenna_type, vendor, mtilt, height, xtxr,local_cell_id,'LTE' as RAT
FROM 
    {sql_lte} a
WHERE
    {where_clause} 

"""
query_nr = f"""
SELECT vendor, site_id, gnodeb_name, sector_name,nr_cell_name as cell_name,nr_du_cell_id as local_cell_id,system,xtxr,ant_type as antenna_type,
'NR' as RAT
FROM {sql_nr} a
WHERE {where_clause}
"""

In [130]:
query_lte

"\nSELECT site, site_id, cell_name, system, sector_name, antenna_type, vendor, mtilt, height, xtxr,local_cell_id,'LTE' as RAT\nFROM \n    lte_wk2521 a\nWHERE\n    site IN ('PTT7028', 'PTT7102', 'PTTC091', 'PTT8518', 'PTT0243', 'PTT2018', 'PTT7982', 'PTTA093', 'PTT7140', 'PTT1020', 'PTT3935', 'PTT7981', 'PTTA078', 'PTTC057', 'PTT2883', 'PTT8610', 'PTT6320', 'PTT3236', 'BKK6661', 'PTTC058', 'PTT0777', 'PTT7983', 'PTTC152', 'PTTA088', 'PTTC042', 'PTTC054', 'PTT7183', 'PTT3868', 'PTT1185', 'PTT1222', 'PTT2918', 'PTT7223', 'PTTC145', 'PTTC148', 'PTT1146', 'PTT8620', 'PTT1175', 'PTT7300', 'PTT6700', 'PTTC047', 'PTT2203', 'PTT8221') \n\n"

### ==== MAPPED SQL ====

In [131]:

query_air = f"""
WITH RankedData AS (
    SELECT 
        LEFT(nodeid, 7) AS site, 
        nodeid, 
        sectorcarrierid, 
        date, 
        digitaltilt,
        ROW_NUMBER() OVER (
            PARTITION BY nodeid, sectorcarrierid
            ORDER BY date DESC
        ) AS RowNum
    FROM eric_air_data
)
SELECT 
    site, 
    nodeid, 
    sectorcarrierid, 
    date, 
    digitaltilt
FROM RankedData 
WHERE {where_clause_1} and RowNum = 1;

"""

query_non_air = f"""
WITH RankedData AS (
    SELECT 
        LEFT(nodeid, 7) AS site, 
        nodeid, 
        userlabel,
        antennaunitgroupid,
        antennanearunitid,
        retsubunitid,
        antennamodelnumber,
        maxtilt,
        mintilt,
        date,
        electricalAntennaTilt,
        ROW_NUMBER() OVER (
            PARTITION BY nodeid, userlabel, antennaunitgroupid, antennanearunitid, retsubunitid, antennamodelnumber, maxtilt, mintilt
            ORDER BY date DESC
        ) AS RowNum
    FROM eric_non_air_data
)
SELECT 
    site, 
    nodeid, 
    userlabel,
    antennaunitgroupid,
    antennanearunitid,
    retsubunitid,
    antennamodelnumber,
    maxtilt,
    mintilt,
    date,
    electricalAntennaTilt
FROM RankedData
WHERE {where_clause_1} and RowNum = 1;


"""

query_hw = f"""
WITH RankedData AS (
    SELECT
        site_name,
        name,
        device_name,
        device_no,
        subunit_no,
        date,
        Actual_tilt,
        ROW_NUMBER() OVER (
            PARTITION BY name, device_name, device_no,subunit_no
            ORDER BY date DESC
        ) AS RowNum
    FROM hwret_data
    WHERE {where_clause_2}
)
SELECT
    site_name,
    a.name,
    a.device_name,
    a.device_no,
    a.subunit_no,
    c.max_tilt,
    c.min_tilt,
    a.date,
    Actual_tilt
FROM RankedData a
LEFT JOIN
retdevicedata_1 c ON concat(a.date,a.NAME,a.Device_Name,a.Device_No,a.subunit_no) = concat(c.date,c.NAME,c.Device_Name,c.Device_No,c.subunit_no)
WHERE {where_clause_2} and RowNum = 1;

"""


### ==== NO MAPPED SQL ====

In [132]:

query_hw_no_map = f"""
SELECT  
    'huawei' AS antenna_type, 
    site_name, 
    a.NAME, 
    a.Device_Name,
    a.Device_No,
    a.subunit_no,
    c.max_tilt,
    c.min_tilt,
    a.date,
    Actual_tilt
FROM hwret_data a
LEFT JOIN
    retdevicedata_1 c ON concat(a.NAME, a.Device_Name, a.Device_No, a.subunit_no) = concat(c.NAME, c.Device_Name, c.Device_No, c.subunit_no)
WHERE a.date >= CURRENT_DATE - INTERVAL '14 days'
  AND {where_clause_2}
GROUP BY 
     
    antenna_type, 
    site_name, 
    a.NAME, 
    a.Device_Name,
    a.Device_No,
    a.subunit_no,
    a.date,
    c.max_tilt,
    c.min_tilt,
    Actual_tilt
"""

query_air_no_map = f"""
SELECT 
     
    'eric_air' AS antenna_type, 
    LEFT(NodeId, 7) AS site_name, 
    NodeId, 
    SectorCarrierId,
    date,
    digitalTilt
    
FROM 
    eric_air_data a

WHERE a.date >= CURRENT_DATE - INTERVAL '14 days' AND {where_clause_1}
GROUP BY 
     
    antenna_type, 
    site_name, 
    NodeId, 
    SectorCarrierId,
    date,
    digitalTilt
"""

query_non_air_no_map = f"""
SELECT 
     
    'eric_non_air' AS antenna_type, 
    LEFT(NodeId, 7) AS site_name, 
    NodeId, 
    CASE 
        WHEN AntennaUnitGroupId ~ '^[0-9]+(\.[0-9]+)?$' THEN
            CASE 
                WHEN POSITION('.' IN AntennaUnitGroupId) > 0 THEN
                    TRIM(TRAILING '.0' FROM AntennaUnitGroupId)
                ELSE 
                    AntennaUnitGroupId
            END
        ELSE 
            AntennaUnitGroupId
    END AS NormalizedAntennaUnitGroupId,  -- Normalizing the AntennaUnitGroupId
    AntennaNearUnitId, 
    RetSubUnitId,
    userLabel,
    AntennaModelNumber,
    maxTilt,
    minTilt,
    date,
    electricalAntennaTilt
FROM 
    eric_non_air_data a
WHERE a.date >= CURRENT_DATE - INTERVAL '14 days' AND {where_clause_1}
GROUP BY 
     
    antenna_type, 
    site_name,
    NodeId, 
    -- Apply the same normalization in the GROUP BY clause
    CASE 
        WHEN AntennaUnitGroupId ~ '^[0-9]+(\.[0-9]+)?$' THEN
            CASE 
                WHEN POSITION('.' IN AntennaUnitGroupId) > 0 THEN
                    TRIM(TRAILING '.0' FROM AntennaUnitGroupId)
                ELSE 
                    AntennaUnitGroupId
            END
        ELSE 
            AntennaUnitGroupId
    END,
    AntennaNearUnitId, 
    RetSubUnitId,
    userLabel,
    AntennaModelNumber,
    maxTilt,
    minTilt,
    date,
    electricalAntennaTilt
"""

query_bfant_tilt = f"""
SELECT 
    a.cell_name,
	a.system,
    a.local_cell_id,
    b.name AS bfant_name,
	b.device_no,
    b.connect_rru_subrack_no,
    c.local_cell_id AS local_cell_id_cellphy,
    b.date,
    b.tilt
FROM {sql_lte} a 
LEFT JOIN cellphytopo c 
    ON CONCAT(a.enodeb_name, a.local_cell_id) = CONCAT(c.name, c.local_cell_id)
LEFT JOIN bfant b 
    ON CONCAT(b.name, b.connect_rru_subrack_no) = CONCAT(c.name, split_part(c.rf_module_information, '-', 2))
WHERE b.date >= CURRENT_DATE - INTERVAL '14 days' AND {where_clause}
GROUP BY a.cell_name,a.system, a.local_cell_id, b.name,b.device_no, b.connect_rru_subrack_no, c.local_cell_id,b.date, b.tilt
"""

query_nr_tilt = f"""
SELECT
	a.nr_cell_name,
	a.system,
	a.nr_du_cell_id,
	b.name AS NRDUCELLTRPBEAM_name,
	b.nr_du_cell_trp_id,
	b.date, b.tilt
FROM {sql_nr} a 
JOIN NRDUCELLTRPBEAM b
    ON CONCAT(a.gnodeb_name, a.nr_du_cell_id) = CONCAT(b.name, b.nr_du_cell_trp_id)
WHERE b.date >= CURRENT_DATE - INTERVAL '14 days' AND {where_clause}
GROUP BY a.nr_cell_name,a.system, a.nr_du_cell_id, b.name,b.nr_du_cell_trp_id,b.date, b.tilt
"""

query_split_tilt = f"""
SELECT
	a.cell_name,
	a.system,
	a.local_cell_id,
	b.name AS SPLITCELL_name,
	b.local_cell_id as SPLITCELL_local_cell_id,
	b.date, cell_beam_tilt
FROM {sql_lte} a 
JOIN SECTORSPLITCELL b
    ON CONCAT(a.enodeb_name, a.local_cell_id) = CONCAT(b.name, b.local_cell_id)
WHERE b.date >= CURRENT_DATE - INTERVAL '14 days' AND {where_clause}
GROUP BY a.cell_name,a.system, a.local_cell_id, b.name,b.local_cell_id,b.date, cell_beam_tilt
"""

In [133]:
query_split_tilt

"\nSELECT\n\ta.cell_name,\n\ta.system,\n\ta.local_cell_id,\n\tb.name AS SPLITCELL_name,\n\tb.local_cell_id as SPLITCELL_local_cell_id,\n\tb.date, cell_beam_tilt\nFROM lte_wk2521 a \nJOIN SECTORSPLITCELL b\n    ON CONCAT(a.enodeb_name, a.local_cell_id) = CONCAT(b.name, b.local_cell_id)\nWHERE b.date >= CURRENT_DATE - INTERVAL '14 days' AND site IN ('PTT7028', 'PTT7102', 'PTTC091', 'PTT8518', 'PTT0243', 'PTT2018', 'PTT7982', 'PTTA093', 'PTT7140', 'PTT1020', 'PTT3935', 'PTT7981', 'PTTA078', 'PTTC057', 'PTT2883', 'PTT8610', 'PTT6320', 'PTT3236', 'BKK6661', 'PTTC058', 'PTT0777', 'PTT7983', 'PTTC152', 'PTTA088', 'PTTC042', 'PTTC054', 'PTT7183', 'PTT3868', 'PTT1185', 'PTT1222', 'PTT2918', 'PTT7223', 'PTTC145', 'PTTC148', 'PTT1146', 'PTT8620', 'PTT1175', 'PTT7300', 'PTT6700', 'PTTC047', 'PTT2203', 'PTT8221')\nGROUP BY a.cell_name,a.system, a.local_cell_id, b.name,b.local_cell_id,b.date, cell_beam_tilt\n"

### ==== RUNNING COMMON DATA =====

In [134]:
conn = connect_postgres(HOST, PORT, DATABASE, USER, PASSWORD)


# Setup project paths
output_dir = os.path.join(OUTPUT_BASE_DIR, folder_name)
os.makedirs(output_dir, exist_ok=True)

# Load and process input


df_lte = fetch_data(query_lte, conn)
df_nr = fetch_data(query_nr, conn)

Connected to PostgreSQL database!


In [135]:
df_lte

,site,site_id,cell_name,system,sector_name,antenna_type,vendor,mtilt,height,xtxr,local_cell_id,rat
0,PTT7028,PTT7028,L23-PTT7028-3C,L2300,PTT7028_C,T2004S6R031,Ericsson,2,24,4T4R,196,LTE
1,PTT0243,PTT0243,L23-PTT0243-1A,L2300,PTT0243_A,AIR 3239 B40,Ericsson,9,38,32T32R,174,LTE
2,PTT0243,PTT0243,L23-PTT0243-1B,L2300,PTT0243_B,AIR 3239 B40,Ericsson,10,38,32T32R,175,LTE
3,PTT0243,PTT0243,L23-PTT0243-1C,L2300,PTT0243_C,ASI4518R53v07,Ericsson,4,38,4T4R,176,LTE
4,PTT0243,PTT0243,L23-PTT0243-1D,L2300,PTT0243_D,ASI4518R53v07,Ericsson,8,38,4T4R,177,LTE
...,...,...,...,...,...,...,...,...,...,...,...,...
1234,PTTC145,PTTC145,L23-PTTC145-3D,L2300,PTTC145_D,ASI4518R53v07,Ericsson,2,24,4T4R,197,LTE
1235,PTTC145,PTTC145,PTTC145R_1NB01_S01,L1800,PTTC145_S01,ASI4518R53v07,Huawei,2,24,4T4R,30,LTE
1236,PTTC145,PTTC145,PTTC145R_1NB01_S02,L1800,PTTC145_S02,AQU4518R59v06,Huawei,2,24,4T4R,31,LTE
1237,PTTC145,PTTC145,PTTC145R_1NB01_S03,L1800,PTTC145_S03,ODDI2-032R20J02-Q,Huawei,2,24,4T4R,32,LTE


### ==== RUNNING MAPPED DATA ====

In [136]:

df_air_1 = fetch_data(query_air, conn)
df_air = df_air_1.pivot(index=['site', 'nodeid', 'sectorcarrierid'], columns='date', values='digitaltilt')
df_air.reset_index(inplace=True)


df_non_air_1 = fetch_data(query_non_air, conn)
df_non_air = df_non_air_1.pivot(index=['site', 'nodeid', 'userlabel','antennaunitgroupid','antennanearunitid','retsubunitid'
                                    ,'antennamodelnumber','mintilt','maxtilt'], columns='date', values='electricalantennatilt')
df_non_air.reset_index(inplace=True)


df_hw_1 = fetch_data(query_hw, conn)
df_hw = df_hw_1.pivot(index=['site_name', 'name', 'device_name', 'device_no','subunit_no','max_tilt','min_tilt'], columns='date', values='actual_tilt')
df_hw.reset_index(inplace=True)

#LTE CELL Normalized
df_lte_cell = lte_cell_normalized(df_lte)

#ERIC_AIR Normalized
df_eric_air = eric_air(df_air, sectorcarrierid_col='sectorcarrierid', nodeid_col='nodeid')
#HWRET Normalized
df_hwret = hwret(df_hw)
df_hwret.rename(columns={'site_name': 'site'}, inplace=True)
#ERIC_NON_AIR Normalized
df_eric_non_air = eric_non_air(df_non_air)

#ERIC_AIR MAP
eric_air_map = pd.merge(
    df_lte_cell,
    df_eric_air,
    on=['site', 'tuning_band', 'sector', 'carrier'],
    how='inner'
    )


eric_air_map['Parameter MO'] = 'SectorCarrier=' + eric_air_map['sectorcarrierid']
eric_air_map['Parameter Name'] = 'digitalTilt'
eric_air_map.sort_values(['site_id', 'tuning_band','sector','carrier'])
eric_air_map.drop_duplicates(inplace=True)

#HWRET MAP
hwret_map = pd.merge(
    df_lte_cell,
    df_hwret,
    on=['site', 'tuning_band', 'sector'],
    how='inner'
)
hwret_map.drop_duplicates(inplace=True)


#ERIC_NON_AIR MAP
eric_non_air_map = pd.merge(
    df_lte_cell,
    df_eric_non_air,
    on=['site', 'tuning_band', 'sector'],
    how='inner'
)
eric_non_air_map.drop_duplicates(inplace=True)



In [137]:
columns_to_include= ['cell_name', 'site_id','system', 'sector_name','rat']
df_MD_LTE_1 = df_lte[columns_to_include]
df_MD_NR_1 = df_nr[columns_to_include]
combined_df = pd.concat([df_MD_LTE_1, df_MD_NR_1], ignore_index=True)

df_cell = df_cell.merge(combined_df, left_on='cell name', right_on='cell_name', how='left')


df_cell

,cell name,cluster name,site_name_1,cell_name,site_id,system,sector_name,rat
0,L23-PTT7028-1D,BMA0319,PTT7028,L23-PTT7028-1D,PTT7028,L2300,PTT7028_D,LTE
1,L23-PTT7028-3A,BMA0319,PTT7028,L23-PTT7028-3A,PTT7028,L2300,PTT7028_A,LTE
2,L23-PTT7102-1D,BMA0319,PTT7102,L23-PTT7102-1D,PTT7102,L2300,PTT7102_D,LTE
3,L23-PTTC091-1C,BMA0319,PTTC091,L23-PTTC091-1C,PTTC091,L2300,PTTC091_C,LTE
4,L23-PTT8518-1C,BMA0319,PTT8518,L23-PTT8518-1C,PTT8518,L2300,PTT8518_C,LTE
...,...,...,...,...,...,...,...,...
215,PTT7140T_2NB01_S52,BMA0319,PTT7140,PTT7140T_2NB01_S52,PTT7140,L2100,PTT7140_S52,LTE
216,PTTA093E_2NB01_S52,BMA0319,PTTA093,PTTA093E_2NB01_S52,PTTA093,L2100,PTTA093_S52,LTE
217,PTTC145T_2NB01_S03,BMA0319,PTTC145,PTTC145T_2NB01_S03,PTTC145,L2100,PTTC145_S03,LTE
218,PTT8221R_1NB02_R01,BMA0319,PTT8221,PTT8221R_1NB02_R01,PTT8221,L1800,PTT8221_R01,LTE


### ==== RUNNING NO MAPPED DATA ====

In [138]:




df_lte['Tuning_Band'] = df_lte['system'].apply(tuning_band_logic)
df_nr['Tuning_Band'] = df_nr['system'].apply(tuning_band_logic)
df_cell['Tuning_Band'] = df_cell['system'].apply(tuning_band_logic)
df_cell_LTE = df_cell[df_cell['rat'].isin(['LTE']) | pd.isna(df_cell['rat']) | ((df_cell['rat'] == 'NR') & (df_cell['system'] == 'NR2600'))]
df_cell_NR = df_cell[df_cell['rat'] == 'NR']

df_lte['seach']= df_lte['site_id']+ df_lte['Tuning_Band']+df_lte['sector_name']
df_nr['seach']= df_nr['site_id']+ df_nr['system']+df_nr['sector_name']
df_cell_LTE['seach']= df_cell_LTE['site_id']+ df_cell_LTE['Tuning_Band']+df_cell_LTE['sector_name']
df_cell_NR['seach']= df_cell_NR['site_id']+ df_cell_NR['system']+df_cell_NR['sector_name']

# Merge df_cell_LTE and df_lte on 'seach', and Cell Name
merged_df_LTE = df_cell_LTE[['seach', 'cell name']].merge(
    df_lte,
    on='seach',
    how='left',  # Use left join to retain all rows from df_cell_LTE
    indicator=True  # Adds a column to show if the match was found
)

# Add a column to indicate if the value was found or not
merged_df_LTE['status'] = merged_df_LTE['_merge'].apply(
    lambda x: 'cannot find in database' if x == 'left_only' else 'found'
)


# Drop the '_merge' and 'seach' columns
merged_df_LTE = merged_df_LTE.drop(columns=['_merge', 'seach'])

# Reset the index
merged_df_LTE = merged_df_LTE.reset_index(drop=True)

# NR
# Merge df_cell_NR and df_nr on 'seach', and Cell Name
merged_df_NR = df_cell_NR[['seach', 'cell name']].merge(
    df_nr,
    on='seach',
    how='left',  # Use left join to retain all rows from df_cell_NR
    indicator=True  # Adds a column to show if the match was found
)

# Add a column to indicate if the value was found or not
merged_df_NR['status'] = merged_df_NR['_merge'].apply(
    lambda x: 'cannot find in database' if x == 'left_only' else 'found'
)

# Drop the '_merge' and 'seach' columns
merged_df_NR = merged_df_NR.drop(columns=['_merge', 'seach'])

# Reset the index
merged_df_NR = merged_df_NR.reset_index(drop=True)

def suggestion(xtxr,vendor, antenna_type, is_lte=True):
    bfant = ['AAU5639', 'AAU5614', 'AAU5636']
    sectorsplitcell = ['AAU5711a', 'AAU5726']
    if vendor == 'Huawei':
        if is_lte: 
            if any(item in antenna_type for item in bfant):
                return 'BFANT'
            elif any(item in antenna_type for item in sectorsplitcell):
                return 'SECTORSPLITCELL'
            else:
                return 'RETSUBUNIT'
        else:
            if xtxr.upper() != '64T64R':
                return 'RETSUBUNIT'
            else:
                return 'NRDUCELLTRPBEAM'
    elif vendor == 'Ericsson':
        if 'AIR' in antenna_type:
            return 'AIR (SectorCarrier)'
        else:
            return 'NON_AIR (ElectricalTilt)'
    else:
        return 'TBD'

# Apply the compacted function
merged_df_LTE['suggestion'] = merged_df_LTE.apply(lambda row: suggestion(row['xtxr'],row['vendor'], row['antenna_type'], is_lte=True), axis=1)
merged_df_NR['suggestion'] = merged_df_NR.apply(lambda row: suggestion(row['xtxr'],row['vendor'], row['antenna_type'], is_lte=False), axis=1)


merged_df_LTE = merged_df_LTE.drop_duplicates()
merged_df_NR = merged_df_NR.drop_duplicates()
merged_df_LTE.rename(columns={'cell name': 'cell_name_remove'}, inplace=True)
merged_df_NR.rename(columns={'cell name': 'cell_name_remove'}, inplace=True)


df_hw_no_map = fetch_data(query_hw_no_map, conn)

# Display the DataFrame
df_hw_no_map.rename(columns={'antenna_type': 'file_type'}, inplace=True)
df_hw_no_map['MO'] = 'RETSUBUNIT'
df_hw_no_map['Parameter'] = 'Tilt'
df_hw_no_map = df_hw_no_map.drop_duplicates()
df_hw_no_map = df_hw_no_map.pivot(index=['file_type', 'site_name','name','device_name','device_no','subunit_no','MO','Parameter','max_tilt','min_tilt'], columns='date', values='actual_tilt')
df_hw_no_map.reset_index(inplace=True)

df_air_no_map = fetch_data(query_air_no_map, conn)
df_air_no_map.rename(columns={'antenna_type': 'file_type'}, inplace=True)
df_air_no_map['MO'] = 'SectorCarrier=' + df_air_no_map['sectorcarrierid'].astype(str)
df_air_no_map['Parameter'] = 'digitalTilt'
df_air_no_map = df_air_no_map.pivot(index=['file_type', 'site_name','nodeid','sectorcarrierid','MO','Parameter'], columns='date', values='digitaltilt')
df_air_no_map.reset_index(inplace=True)


df_non_air_no_map = fetch_data(query_non_air_no_map, conn)
df_non_air_no_map.rename(columns={'antenna_type': 'file_type'}, inplace=True)
# Columns to change to int
change_to_int = [ 'antennanearunitid', 'retsubunitid']

# Convert to numeric (float), then to integer
df_non_air_no_map[change_to_int] = df_non_air_no_map[change_to_int].apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)
df_non_air_no_map['MO'] = 'AntennaUnitGroup='+ df_non_air_no_map['normalizedantennaunitgroupid'].astype(str) +',AntennaNearUnit=' + df_non_air_no_map['antennanearunitid'].astype(str) +', RetSubUnit='+ df_non_air_no_map['retsubunitid'].astype(str)
df_non_air_no_map['Parameter'] = 'electricalAntennaTilt'
df_non_air_no_map = df_non_air_no_map.pivot(index=[ 'file_type', 'site_name','nodeid','normalizedantennaunitgroupid','antennanearunitid','retsubunitid'
                             ,'userlabel','antennamodelnumber','mintilt','maxtilt','MO','Parameter'], columns='date', values='electricalantennatilt')
df_non_air_no_map.reset_index(inplace=True)


df_bfant_tilt = fetch_data(query_bfant_tilt, conn)
df_bfant_tilt = df_bfant_tilt.pivot(index=['cell_name', 'system', 'local_cell_id','bfant_name','device_no',
                                           'connect_rru_subrack_no','local_cell_id_cellphy'], columns='date', values='tilt')
df_bfant_tilt.reset_index(inplace=True)


df_nr_tilt = fetch_data(query_nr_tilt, conn)
df_nr_tilt = df_nr_tilt.pivot(index=['nr_cell_name', 'system', 'nr_du_cell_id','nrducelltrpbeam_name','nr_du_cell_trp_id'
                                           ], columns='date', values='tilt')
df_nr_tilt.reset_index(inplace=True)


df_split_tilt = fetch_data(query_split_tilt, conn)
df_split_tilt = df_split_tilt.pivot(index=['cell_name', 'system', 'local_cell_id','splitcell_name','splitcell_local_cell_id'
                                           ], columns='date', values='cell_beam_tilt')
df_split_tilt.reset_index(inplace=True)

C:\Users\User\AppData\Local\Temp\ipykernel_2592\598853818.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cell_NR['seach']= df_cell_NR['site_id']+ df_cell_NR['system']+df_cell_NR['sector_name']


### ==== OUTPUT MAPPED DATA =====

In [139]:
hwret_map.to_csv(os.path.join(output_dir, f'{cluster_name}_hwret_map.csv'), index=False)
eric_air_map.to_csv(os.path.join(output_dir, f'{cluster_name}_eric_air_map.csv'), index=False)
eric_non_air_map.to_csv(os.path.join(output_dir, f'{cluster_name}_eric_non_air_map.csv'), index=False)
csv_files = [
    f'{cluster_name}_hwret_map.csv',
    f'{cluster_name}_eric_air_map.csv',
    f'{cluster_name}_eric_non_air_map.csv'
    ]
zip_file_path = os.path.join(output_dir, f'{cluster_name}_files_map.zip')

# Create the list of file paths
file_paths = [os.path.join(output_dir, file) for file in csv_files]

# Create the .zip archive
with zipfile.ZipFile(zip_file_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in file_paths:
        if os.path.exists(file):
            zipf.write(file, os.path.basename(file))  # Add file to ZIP with its base name
        else:
            print(f"File {file} does not exist.")

print(f"ZIP archive created at: {zip_file_path}")

# Clean up (delete) the CSV files after zipping them
for file in file_paths:
    if os.path.exists(file):
        os.remove(file)
        print(f"Deleted: {file}")
    else:
        print(f"File {file} does not exist for deletion.")

ZIP archive created at: D:/D&T Project/CR Preparing/BMA0319\BMA0319_files_map.zip
Deleted: D:/D&T Project/CR Preparing/BMA0319\BMA0319_hwret_map.csv
Deleted: D:/D&T Project/CR Preparing/BMA0319\BMA0319_eric_air_map.csv
Deleted: D:/D&T Project/CR Preparing/BMA0319\BMA0319_eric_non_air_map.csv


### ==== OUTPUT NO MAPPED DATA =====

In [140]:
merged_df_LTE.to_csv(os.path.join(output_dir, f'Cell_LTE_result_{cluster_name}.csv'), index=False)
merged_df_NR.to_csv(os.path.join(output_dir, f'Cell_NR_result_{cluster_name}.csv'), index=False)
df_hw_no_map.to_csv(os.path.join(output_dir, f'{cluster_name}_hw.csv'), index=False)
df_air_no_map.to_csv(os.path.join(output_dir, f'{cluster_name}_air.csv'), index=False)
df_non_air_no_map.to_csv(os.path.join(output_dir, f'{cluster_name}_non_air.csv'), index=False)
df_bfant_tilt.to_csv(os.path.join(output_dir, f'{cluster_name}_bfant_tilt.csv'), index=False)
df_nr_tilt.to_csv(os.path.join(output_dir, f'{cluster_name}_nr_tilt.csv'), index=False)
df_split_tilt.to_csv(os.path.join(output_dir, f'{cluster_name}_split_tilt.csv'), index=False)
#df_RETSUBUNIT.to_csv(os.path.join(output_dir, f'{cluster_name}_RETSUBUNIT_map.csv'), index=False)
# List of CSV files
csv_files = [
    f'Cell_LTE_result_{cluster_name}.csv',
    f'Cell_NR_result_{cluster_name}.csv',
    f'{cluster_name}_hw.csv',
    f'{cluster_name}_air.csv',
    f'{cluster_name}_non_air.csv',
    f'{cluster_name}_bfant_tilt.csv',
    f'{cluster_name}_nr_tilt.csv',
    f'{cluster_name}_split_tilt.csv'
]

# Path for the zip file
zip_file_path = os.path.join(output_dir, f'{cluster_name}_files_1.zip')

# Create the list of file paths
file_paths = [os.path.join(output_dir, file) for file in csv_files]

# Create the .zip archive
with zipfile.ZipFile(zip_file_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in file_paths:
        if os.path.exists(file):
            zipf.write(file, os.path.basename(file))  # Add file to ZIP with its base name
        else:
            print(f"File {file} does not exist.")




print(f"ZIP archive created at: {zip_file_path}")

# Clean up (delete) the CSV files after zipping them
for file in file_paths:
    if os.path.exists(file):
        os.remove(file)
        print(f"Deleted: {file}")
    else:
        print(f"File {file} does not exist for deletion.")


ZIP archive created at: D:/D&T Project/CR Preparing/BMA0319\BMA0319_files_1.zip
Deleted: D:/D&T Project/CR Preparing/BMA0319\Cell_LTE_result_BMA0319.csv
Deleted: D:/D&T Project/CR Preparing/BMA0319\Cell_NR_result_BMA0319.csv
Deleted: D:/D&T Project/CR Preparing/BMA0319\BMA0319_hw.csv
Deleted: D:/D&T Project/CR Preparing/BMA0319\BMA0319_air.csv
Deleted: D:/D&T Project/CR Preparing/BMA0319\BMA0319_non_air.csv
Deleted: D:/D&T Project/CR Preparing/BMA0319\BMA0319_bfant_tilt.csv
Deleted: D:/D&T Project/CR Preparing/BMA0319\BMA0319_nr_tilt.csv
Deleted: D:/D&T Project/CR Preparing/BMA0319\BMA0319_split_tilt.csv
